# DATA TRANSFORMATIONS

In [ ]:
select top(3) * from process_mining.dbo.O2C_Entities_Orders

In [ ]:
select top(3) * from process_mining.dbo.O2C_Master_Data_Customers

In [ ]:
select top(3) * from process_mining.dbo.O2C_Master_Data_Order_Types

In [ ]:
select top(3) * from process_mining.dbo.O2C_Master_Data_Users

In [ ]:
select top(3) * from process_mining.dbo.O2C_Entities_Delveries

# CASE LEVEL ATTRIBUTES

In [ ]:
select top(3) 
Order_ID as Case_ID, 
o.Customer_ID, 
Customer_Name,
Customer_Country,
Customer_Type,
Order_Value, 
Currency, 
concat(o.Order_Type, ' - ', Order_Type_Description) as Order_Type,
Company_Code, 
Delivery_Block, 
Billing_Block
from process_mining.dbo.O2C_Entities_Orders o
left join process_mining.dbo.O2C_Master_Data_Customers c
on o.Customer_ID = c.Customer_ID
left join process_mining.dbo.O2C_Master_Data_Order_Types ot 
on o.Order_Type = ot.Order_Type

# EVENT LEVEL ATTRIBUTES

## CREATE SALES ORDER

In [28]:

select
Order_ID as Case_ID, 
o.Customer_ID, 
Customer_Name,
Customer_Country,
Customer_Type,
Order_Value, 
Currency, 
concat(o.Order_Type, ' - ', Order_Type_Description) as Order_Type,
Company_Code, 
Delivery_Block, 
Billing_Block,
'Create Sales Order' as Event_Type,

CAST(
        RIGHT(Creation_Date, 4) + '-' + 
        SUBSTRING(Creation_Date, 4, 2) + '-' + 
        LEFT(Creation_Date, 2) + ' ' + 
        Creation_Time 
        AS datetime
    ) AS Event_Time,
o.[User],
Department,
User_Type,
Order_ID as Object_ID,
'Sales order' as Object_Type,
'Main' as Event_Category
into process_mining.dbo.O2C_Events_Create_sales_order
from process_mining.dbo.O2C_Entities_Orders o
left join process_mining.dbo.O2C_Master_Data_Customers c
on o.Customer_ID = c.Customer_ID
left join process_mining.dbo.O2C_Master_Data_Order_Types ot 
on o.Order_Type = ot.Order_Type
left join process_mining.dbo.O2C_Master_Data_Users u
on o.[User] = u.User_ID
where Order_ID is not NULL

(1902 rows affected)

Total execution time: 00:00:00.363

In [ ]:
select * from process_mining.dbo.O2C_Events_Create_sales_order

## REQUESTED DELIVERY DATE

In [46]:
select 
o.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
concat(o.Order_Type, ' - ', Order_Type_Description) as Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Requested Delivery Date' as Event_Type,
CAST(
        RIGHT(Creation_Date, 4) + '-' + 
        SUBSTRING(Creation_Date, 4, 2) + '-' + 
        LEFT(Creation_Date, 2) + ' ' + 
        '23:59:59' 
        AS datetime
    ) AS Event_Time,
o.[User],
Department,
User_Type,
Order_ID as Object_ID,
'Sales order' as Object_Type,
'Main' as Event_Category
into process_mining.dbo.O2C_Events_Requested_delivery_date
from process_mining.dbo.O2C_Entities_Orders o
left join process_mining.dbo.O2C_Master_Data_Customers c
on o.Customer_ID = c.Customer_ID
left join process_mining.dbo.O2C_Master_Data_Order_Types ot 
on o.Order_Type = ot.Order_Type
left join process_mining.dbo.O2C_Master_Data_Users u
on o.[User] = u.User_ID

where Requested_Delivery_Date is not NULL

(1902 rows affected)

Total execution time: 00:00:00.317

In [ ]:
select * from process_mining.dbo.O2C_Events_Requested_delivery_date

## CREATE DELIVERY

In [50]:
select 
d.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Create Delivery' as Event_Type,
CAST(
        RIGHT(Created_Date, 4) + '-' + 
        SUBSTRING(Created_Date, 4, 2) + '-' + 
        LEFT(Created_Date, 2) + ' ' + 
        Created_time 
        AS datetime
    ) AS Event_Time,
d.[User],
Department,
User_Type,
Delivery_ID as Object_ID,
'Delivery' as Object_Type,
'Main' as Event_Category
into process_mining.dbo.O2C_Events_Create_delivery

from process_mining.dbo.O2C_Entities_Delveries d

left join process_mining.dbo.O2C_Master_Data_Users u
on d.[User] = u.User_ID

where Delivery_ID is not NULL

(616 rows affected)

Total execution time: 00:00:00.313

## CREATE FREIGHT ORDER

adat duplikaciók vannak, ahol egy orderre több freight order is készült (658 -- 661)

cause: USER - 07901

In [11]:
select count(*) from process_mining.dbo.O2C_Entities_FreightOrders

select distinct Created_by, count(*) from process_mining.dbo.O2C_Entities_FreightOrders group by Created_by

select * from process_mining.dbo.O2C_Entities_FreightOrders where Created_by = 'USER - 07901'



(1 row affected)

(8 rows affected)

(1 row affected)

Total execution time: 00:00:00.011

(No column name)
658


Created_by,(No column name)
NULL,8
USER - 03676,577
USER - 05664,39
USER - 05724,29
USER - 05732,1
USER - 07760,1
USER - 07901,1
USER - 10696,2


Freight_Order_ID,Created_at,Creation_time,Order_ID,Created_by
FREIGHT - 00209,03/10/2023,09:21:13,Order - 0233,USER - 07901


In [12]:
select *
from process_mining.dbo.O2C_Entities_FreightOrders f
left join process_mining.dbo.O2C_Master_Data_Users u
on Created_by = u.User_ID
where Created_by = 'USER - 07901'


(1 row affected)

Total execution time: 00:00:00.027

Freight_Order_ID,Created_at,Creation_time,Order_ID,Created_by,User_Name,User_ID,Department,User_Type
FREIGHT - 00209,03/10/2023,09:21:13,Order - 0233,USER - 07901,User Name 7901,USER - 07901,Department 1776,Manual


In [86]:
select * from process_mining.dbo.O2C_Master_Data_Users where User_ID = 'USER - 07901'

(4 rows affected)

Total execution time: 00:00:00.016

User_Name,User_ID,Department,User_Type
User Name 7901,USER - 07901,Department 1776,Manual
User Name 7901,USER - 07901,Department 2822,Manual
User Name 7901,USER - 07901,Department 2822,Manual
User Name 7901,USER - 07901,Department 2822,Manual


In [14]:
select 
f.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Create Freight Order' as Event_Type,
CAST(
        RIGHT(Created_at, 4) + '-' + 
        SUBSTRING(Created_at, 4, 2) + '-' + 
        LEFT(Created_at, 2) + ' ' + 
        Creation_time
        AS datetime
    ) AS Event_Time,
Created_by as [User],
Department,
User_Type,
Freight_Order_ID as Object_ID,
'Freight Order' as Object_Type,
'Main' as Event_Category

into process_mining.dbo.O2C_Events_Create_freight_order

from process_mining.dbo.O2C_Entities_FreightOrders f

left join process_mining.dbo.O2C_Master_Data_Users u
on Created_by = u.User_ID

where Freight_Order_ID is not NULL


(658 rows affected)

Total execution time: 00:00:00.299

## Create Goods Issue

In [94]:
select count(*) from process_mining.dbo.O2C_Entities_GoodsIssued
select count(*) from process_mining.dbo.O2C_Entities_GoodsIssued where Goods_Issue_ID IS NULL
select top(3) * from process_mining.dbo.O2C_Entities_GoodsIssued

(1 row affected)

(1 row affected)

(3 rows affected)

Total execution time: 00:00:00.028

(No column name)
1034


(No column name)
7


Order_ID,Goods_Issue_ID,Creation_date,Creation_time
Order - 0001,GI - 000001,10/05/2023,16:12:27
Order - 0001,GI - 000002,10/05/2023,16:12:27
Order - 0001,GI - 000003,10/05/2023,16:12:27


In [95]:
select 
g.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Create Goods Issue' as Event_Type,
CAST(
        RIGHT(Creation_date, 4) + '-' + 
        SUBSTRING(Creation_date, 4, 2) + '-' + 
        LEFT(Creation_date, 2) + ' ' + 
        Creation_time
        AS datetime
    ) AS Event_Time,
'No user information' [User],
'' Department,
'' User_Type,
Goods_Issue_ID as Object_ID,
'Goods Issue is created' as Object_Type,
'Main' as Event_Category

into process_mining.dbo.O2C_Events_Goods_issue_created

from process_mining.dbo.O2C_Entities_GoodsIssued g

where Goods_Issue_ID is not NULL

(1027 rows affected)

Total execution time: 00:00:00.021

## Invoices

In [97]:
select count(*) from process_mining.dbo.O2C_Entities_Invoices
select count(*) from process_mining.dbo.O2C_Entities_Invoices where Invoice_ID IS NULL
select top(3) * from process_mining.dbo.O2C_Entities_Invoices

(1 row affected)

(1 row affected)

(3 rows affected)

Total execution time: 00:00:00.020

(No column name)
644


(No column name)
0


Order_ID,Invoice_Date,Invoice_ID,Created_by,Payment_date,Payment_booked_by,Customer_ID
Order - 0001,10/05/2023 22:11,Invoice - 001,USER - 07897,2023-09-27,USER - 07897,CUST - 001
Order - 0002,15/05/2023 22:04,Invoice - 002,USER - 07897,2023-07-14,USER - 07897,CUST - 002
Order - 0012,25/05/2023 22:00,Invoice - 003,USER - 07897,2023-06-21,USER - 07897,CUST - 005


In [113]:
select 
i.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Create Invoice' as Event_Type,
CAST(
        SUBSTRING(Invoice_Date, 7, 4) + '-' + 
        SUBSTRING(Invoice_Date, 4, 2) + '-' + 
        LEFT(Invoice_Date, 2) + ' ' + 
        SUBSTRING(Invoice_Date, 12, 16)
        AS datetime
    ) AS Event_Time,
Created_by [User],
Department,
User_Type,
Invoice_ID as Object_ID,
'Create Invoice' as Object_Type,
'Main' as Event_Category

into process_mining.dbo.O2C_Events_Create_invoice

from process_mining.dbo.O2C_Entities_Invoices i

left join process_mining.dbo.O2C_Master_Data_Users u
on Created_by = u.User_ID

where Invoice_ID is not NULL

: Msg 2714, Level 16, State 6, Line 1
There is already an object named 'O2C_Events_Create_invoice' in the database.

Total execution time: 00:00:00.006

## Invoice is paid

In [115]:
select 
i.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Invoice is paid' as Event_Type,
cast(Payment_date as datetime) + cast('23:59:59' as datetime) AS Event_Time,
Payment_booked_by [User],
Department,
User_Type,
Invoice_ID as Object_ID,
'Invoice is paid' as Object_Type,
'Main' as Event_Category

into process_mining.dbo.O2C_Events_Invoice_is_paid

from process_mining.dbo.O2C_Entities_Invoices i

left join process_mining.dbo.O2C_Master_Data_Users u
on Payment_booked_by = u.User_ID

where Payment_date is not NULL

(356 rows affected)

Total execution time: 00:00:00.304

# CHANGE EVENTS

In [118]:
SELECT distinct [Object] from process_mining.dbo.O2C_ChangeLog

(4 rows affected)

Total execution time: 00:00:00.014

Object
Delivery document
Freight order
Invoice
Sales document


In [4]:
select count(*) from process_mining.dbo.O2C_ChangeLog
where OBJECT='Sales document'

SELECT * from process_mining.dbo.O2C_ChangeLog

(1 row affected)

(163 rows affected)

Total execution time: 00:00:00.036

(No column name)
46


Object,Object_ID,Timestamp,User
Delivery document,Delivery - 0001,10/05/2023 13:36,USER - 03676
Sales document,Order - 0001,10/05/2023 13:38,USER - 03676
Freight order,FREIGHT - 00001,10/05/2023 13:54,USER - 03676
Delivery document,Delivery - 0001,10/05/2023 15:33,USER - 07760
Freight order,FREIGHT - 00003,15/05/2023 11:04,USER - 03676
Delivery document,Delivery - 0002,15/05/2023 15:52,USER - 10709
Freight order,FREIGHT - 00005,25/05/2023 11:58,USER - 03676
Delivery document,Delivery - 0003,25/05/2023 13:14,USER - 10699
Freight order,FREIGHT - 00007,25/05/2023 13:14,USER - 03676
Freight order,FREIGHT - 00009,25/05/2023 13:32,USER - 03676


## Change sales order

In [22]:
select Object_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Sales order is changed' as Event_Type,

CAST(
    SUBSTRING([Timestamp], 7, 4) + '-' +  -- yyyy
    SUBSTRING([Timestamp], 4, 2) + '-' +  -- MM
    left([Timestamp], 2) + ' ' +  -- dd
    SUBSTRING([Timestamp], 12, 5)        -- HH:mm
AS datetime) AS Event_Time,

[User],
Department,
User_Type,
Object_ID,
'Sales order' as Object_Type,
'Change' as Event_Category

into process_mining.dbo.O2C_Events_Change_Sales_order

from process_mining.dbo.O2C_ChangeLog cl

left join process_mining.dbo.O2C_Master_Data_Users u
on cl.[User] = u.User_ID

where OBJECT='Sales document'

(46 rows affected)

Total execution time: 00:00:00.026

## Change delivery

In [125]:
select count(*) from process_mining.dbo.O2C_ChangeLog
where OBJECT='Delivery document'

(1 row affected)

Total execution time: 00:00:00.004

(No column name)
55


In [9]:
select
d.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Delivery is changed' as Event_Type,

CAST(
    SUBSTRING([Timestamp], 7, 4) + '-' +  -- yyyy
    SUBSTRING([Timestamp], 4, 2) + '-' +  -- MM
    left([Timestamp], 2) + ' ' +  -- dd
    SUBSTRING([Timestamp], 12, 5)        -- HH:mm
AS datetime) AS Event_Time,

cl.[User],
Department,
User_Type,
Object_ID,
'Delivery' as Object_Type,
'Change' as Event_Category

into process_mining.dbo.O2C_Events_Change_Delivery

from process_mining.dbo.O2C_ChangeLog cl

left join process_mining.dbo.O2C_Entities_Delveries d
on cl.Object_ID = d.Delivery_ID

left join process_mining.dbo.O2C_Master_Data_Users u
on cl.[User] = u.User_ID

where OBJECT='Delivery document' and Order_ID is not NULL

(55 rows affected)

Total execution time: 00:00:00.299

## Change freight order

DUPLIKATUMOKAT TO HANDLE!!!

In [15]:
select count(*) from process_mining.dbo.O2C_ChangeLog
where OBJECT='Freight order'

(1 row affected)

Total execution time: 00:00:00.007

(No column name)
43


In [17]:
select
f.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Freight order is changed' as Event_Type,

CAST(
    SUBSTRING([Timestamp], 7, 4) + '-' +  -- yyyy
    SUBSTRING([Timestamp], 4, 2) + '-' +  -- MM
    left([Timestamp], 2) + ' ' +  -- dd
    SUBSTRING([Timestamp], 12, 5)        -- HH:mm
AS datetime) AS Event_Time,

cl.[User],
Department,
User_Type,
Object_ID,
'Freight order' as Object_Type,
'Change' as Event_Category

into process_mining.dbo.O2C_Events_Change_Freight_order

from process_mining.dbo.O2C_ChangeLog cl

left join process_mining.dbo.O2C_Entities_FreightOrders f
on cl.Object_ID = f.Freight_Order_ID

left join process_mining.dbo.O2C_Master_Data_Users u
on cl.[User] = u.User_ID

where OBJECT='Freight order' and Order_ID is not NULL

(43 rows affected)

Total execution time: 00:00:00.303

## CREATE INVOICE

In [133]:
select count(*) from process_mining.dbo.O2C_ChangeLog
where OBJECT='Invoice'

(1 row affected)

Total execution time: 00:00:00.003

(No column name)
19


In [13]:
select
i.Order_ID as Case_ID, 
'' Customer_ID, 
'' Customer_Name,
'' Customer_Country,
'' Customer_Type,
'' Order_Value, 
'' Currency, 
'' Order_Type,
'' Company_Code, 
'' Delivery_Block, 
'' Billing_Block,
'Invoice is changed' as Event_Type,

CAST(
    SUBSTRING([Timestamp], 7, 4) + '-' +  -- yyyy
    SUBSTRING([Timestamp], 4, 2) + '-' +  -- MM
    left([Timestamp], 2) + ' ' +  -- dd
    SUBSTRING([Timestamp], 12, 5)        -- HH:mm
AS datetime) AS Event_Time,


cl.[User],
Department,
User_Type,
Object_ID,
'Invoice' as Object_Type,
'Change' as Event_Category

into process_mining.dbo.O2C_Events_Change_Invoice

from process_mining.dbo.O2C_ChangeLog cl

left join process_mining.dbo.O2C_Entities_Invoices i
on cl.Object_ID = i.Invoice_ID

left join process_mining.dbo.O2C_Master_Data_Users u
on cl.[User] = u.User_ID

where OBJECT='Invoice' and Order_ID is not NULL

(19 rows affected)

Total execution time: 00:00:00.299

# FINAL EVENT TABLE

In [18]:
USE process_mining;
GO

SELECT 
    o.name AS Object_Name,
    o.type_desc AS Object_Type,
    COUNT(c.name) AS Column_Count
FROM 
    sys.objects o
JOIN 
    sys.columns c ON o.object_id = c.object_id
WHERE 
    o.name IN (
        'O2C_Events_Create_Sales_order',
        'O2C_Events_Requested_delivery_date',
        'O2C_Events_Create_delivery',
        'O2C_Events_Create_freight_order',
        'O2C_Events_Goods_issue_created',
        'O2C_Events_Create_invoice',
        'O2C_Events_Invoice_is_paid',
        'O2C_Events_Change_Sales_order',
        'O2C_Events_Change_Delivery',
        'O2C_Events_Change_Freight_order',
        'O2C_Events_Change_Invoice'
    )
GROUP BY 
    o.name, o.type_desc
ORDER BY 
    o.name;

Commands completed successfully.

(11 rows affected)

Total execution time: 00:00:00.050

Object_Name,Object_Type,Column_Count
O2C_Events_Change_Delivery,USER_TABLE,19
O2C_Events_Change_Freight_order,USER_TABLE,19
O2C_Events_Change_Invoice,USER_TABLE,19
O2C_Events_Change_Sales_order,USER_TABLE,19
O2C_Events_Create_delivery,USER_TABLE,19
O2C_Events_Create_freight_order,USER_TABLE,19
O2C_Events_Create_invoice,USER_TABLE,19
O2C_Events_Create_sales_order,USER_TABLE,19
O2C_Events_Goods_issue_created,USER_TABLE,19
O2C_Events_Invoice_is_paid,USER_TABLE,19


In [21]:
SELECT * 
INTO process_mining.dbo.O2C_Eventlog

FROM(
SELECT * FROM process_mining.dbo.O2C_Events_Create_Sales_order
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Requested_delivery_date
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Create_delivery
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Create_freight_order
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Goods_issue_created
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Create_invoice
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Invoice_is_paid
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Change_Sales_order
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Change_Delivery
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Change_Freight_order
UNION ALL SELECT * FROM  process_mining.dbo.O2C_Events_Change_Invoice
) as Combined_events

(7268 rows affected)

Total execution time: 00:00:00.188

In [1]:
select *  from process_mining.dbo.O2C_Eventlog

(7268 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.321

Case_ID,Customer_ID,Customer_Name,Customer_Country,Customer_Type,Order_Value,Currency,Order_Type,Company_Code,Delivery_Block,Billing_Block,Event_Type,Event_Time,User,Department,User_Type,Object_ID,Object_Type,Event_Category
Order - 0523,CUST - 014,Customer 14,DE,External,117784.08,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-12-19 14:33:39.000,USER - 10897,Department 2178,Manual,Order - 0523,Sales order,Main
Order - 0524,CUST - 014,Customer 14,DE,External,14989.68,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-12-19 14:37:04.000,USER - 10897,Department 2178,Manual,Order - 0524,Sales order,Main
Order - 0155,CUST - 014,Customer 14,DE,External,-890.64,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 15:42:28.000,USER - 10897,Department 2178,Manual,Order - 0155,Sales order,Main
Order - 0156,CUST - 014,Customer 14,DE,External,-5250.96,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 15:50:20.000,USER - 10897,Department 2178,Manual,Order - 0156,Sales order,Main
Order - 0157,CUST - 014,Customer 14,DE,External,-10361.28,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 15:55:14.000,USER - 10897,Department 2178,Manual,Order - 0157,Sales order,Main
Order - 0158,CUST - 014,Customer 14,DE,External,-19299.12,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 15:59:05.000,USER - 10897,Department 2178,Manual,Order - 0158,Sales order,Main
Order - 0159,CUST - 014,Customer 14,DE,External,-39851.28,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 16:07:02.000,USER - 10897,Department 2178,Manual,Order - 0159,Sales order,Main
Order - 0152,CUST - 019,Customer 19,AT,External,20.4,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2023-08-16 14:29:54.000,USER - 10897,Department 2178,Manual,Order - 0152,Sales order,Main
Order - 1391,CUST - 019,Customer 19,AT,External,51.6,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2024-07-10 15:14:34.000,USER - 10897,Department 2178,Manual,Order - 1391,Sales order,Main
Order - 1367,CUST - 026,Customer 26,DE,External,45938.02,EUR,ST - Services / Tooling,Company 1,NULL,NULL,Create Sales Order,2024-07-05 15:43:57.000,USER - 10897,Department 2178,Manual,Order - 1367,Sales order,Main
